In [12]:
import redis
import json
import socket

REDIS_HOST = "localhost"
REDIS_PORT = 6379
REDIS_DB = 0

# Setting Server

In [13]:
# ตั้งค่าที่อยู่เซิร์ฟเวอร์
HOST = '127.0.0.1'  # ใช้ localhost
PORT = 5001         # พอร์ตของ TCP server

# ส่งคำสั่งไปยังเซิร์ฟเวอร์
def send_command(command):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))
        s.sendall((command + '\n').encode('utf-8'))  # ส่งคำสั่ง
        data = s.recv(4096)  # รับผลลัพธ์กลับมา
        print(f'Received: {data.decode()}')

In [3]:
# Send cmd json form
# ส่งคำสั่งไปยังเซิร์ฟเวอร์
import json
import socket

# ตั้งค่าที่อยู่เซิร์ฟเวอร์
HOST = '127.0.0.1'  # ใช้ localhost
PORT = 5001         # พอร์ตของ TCP server

# ✅ ส่ง JSON Command ไปยัง Server
def send_command(command_dict):
    """ ส่งคำสั่งไปยังเซิร์ฟเวอร์ในรูปแบบ JSON """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))

        # ✅ แปลง dict เป็น JSON และส่งไป
        json_command = json.dumps(command_dict) + '\n'
        s.sendall(json_command.encode('utf-8'))  

        # ✅ รับข้อมูลตอบกลับจาก Server
        response = s.recv(4096)
        response_data = response.decode()
        
        try:
            # ✅ แปลง JSON ตอบกลับเป็น Dict
            response_json = json.loads(response_data)
        except json.JSONDecodeError:
            response_json = {"error": "Invalid JSON response from server"}

        print(f"📌 Server Response: {json.dumps(response_json, indent=4)}")
        return response_json


In [14]:
# ✅ Import Libraries
import socket
import json

# ✅ ตั้งค่าที่อยู่เซิร์ฟเวอร์
HOST = '127.0.0.1'  # ใช้ localhost
PORT = 5001         # พอร์ตของ TCP server

def send_command(command_dict):
    """ ส่ง JSON Command ไปยัง Server และรับ JSON Response """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))

        # ✅ แปลง Python Dict เป็น JSON String ก่อนส่ง
        command_json = json.dumps(command_dict)  
        s.sendall((command_json + '\n').encode('utf-8'))

        # ✅ รับข้อมูลจาก Server
        data = s.recv(4096).decode('utf-8').strip()

        try:
            # ✅ เช็คว่า Response เป็น JSON จริงหรือไม่
            response = json.loads(data)  
            print("📌 Received JSON Response:", json.dumps(response, indent=4))
            return response
        except json.JSONDecodeError:
            print("❌ Received non-JSON response:", data)
            return {"error": "Invalid response format"}


# Training Request

In [17]:
# ทดสอบคำสั่งต่างๆ
# ฝึกโมเดล
response = send_command('train|classify')  

# รับผลลัพธ์จาก Server

# ✅ รอ 5 วินาทีก่อนปิดการเชื่อมต่อ

# ประเมินโมเดล
# ส่งคำสั่ง evaluate พร้อมกับ path ของ model

Received: {"status": "success", "data": {"status": "success", "data": {"message": "Training completed successfully", "mode": "classify", "model_path": "runs\\classify\\train2\\weights\\best.pt", "result_dir": "runs\\classify\\train2", "saved_path": "C:/MediScan/models/model_20250226_045802.pt"}}}



In [15]:
# json cmd
response = send_command({
    "command": "train",
    "mode": "classify",
    "p_id": "project_001"
})


📌 Received JSON Response: {
    "error": "Unexpected error: \u274c data.yaml not found at C:\\MedSight_Project\\proj_001\\data.yaml"
}


# Evaluate

In [29]:
#runable
# evaluate_command = 'evaluate|classify|./runs/classify/train20/weights/best.pt'
# เปลี่ยนเป็น detect, segment, classify ได้
mode = "detect"  
model_path = "detect.pt"
eval_type = 'val'
evaluate_command = f"evaluate|{eval_type}|{mode}|{model_path}"
send_command(evaluate_command)

Received: {"status": "success", "data": {"message": "YOLO val evaluation completed successfully", "metrics": {"mAP50": 0.8308250000000001, "mAP50-95": 0.5984925000000002, "overall_precision": 1.0, "overall_recall": 0.6666666666666666, "overall_f1_score": 0.8, "overall_accuracy": 0.6666666666666666, "class_details": {"cancer": {"precision": 1.0, "recall": 0.3333333333333333, "f1_score": 0.5, "accuracy": 0.3333333333333333}, "non-cancer": {"precision": 1.0, "recall": 1.0, "f1_score": 1.0, "accuracy": 1.0}}}}}



In [10]:
# ✅ ตั้งค่า Evaluate Command
evaluate_command = {
    "command": "evaluate",
    # "eval_type": "val",  # เลือก val หรือ test
    "mode": "detect",    # เปลี่ยนเป็น detect, segment, classify ได้
    "model_name": "detect.pt"
}

print(f"🚀 Sending Evaluate Command: {json.dumps(evaluate_command, indent=4)}")
response = send_command(evaluate_command)


🚀 Sending Evaluate Command: {
    "command": "evaluate",
    "mode": "detect",
    "model_name": "detect.pt"
}
📌 Received JSON Response: {
    "status": "success",
    "data": {
        "message": "YOLO test evaluation completed successfully",
        "metrics": {
            "mAP50": 0.4975,
            "mAP50-95": 0.33599999999999997,
            "overall_precision": 0.5,
            "overall_recall": 0.5,
            "overall_f1_score": 0.5,
            "overall_accuracy": 0.5,
            "class_details": {
                "cancer": {
                    "precision": 0.0,
                    "recall": 0.0,
                    "f1_score": 0,
                    "accuracy": 0
                },
                "non-cancer": {
                    "precision": 1.0,
                    "recall": 1.0,
                    "f1_score": 1.0,
                    "accuracy": 1.0
                }
            }
        }
    }
}


# Deploy

In [24]:
model_name = "best.pt"  # 🔄 เปลี่ยนเป็นชื่อโมเดลที่ต้องการโหลด
deploy_command = f"deploy|{model_name}"
print(f"🚀 Sending Deploy Command: {deploy_command}")
send_command(deploy_command)

🚀 Sending Deploy Command: deploy|best.pt
Received: {"status": "success", "message": "Model 'best.pt' deployed successfully"}



In [42]:
# ✅ ตั้งค่า Deploy Command
deploy_command = {
    "command": "deploy",
    "model_name": "segment.pt"  # เปลี่ยนเป็นชื่อโมเดลที่ต้องการโหลด
}

print(f"🚀 Sending Deploy Command: {json.dumps(deploy_command, indent=4)}")
response = send_command(deploy_command)


🚀 Sending Deploy Command: {
    "command": "deploy",
    "model_name": "segment.pt"
}
📌 Server Response: {
    "status": "success",
    "message": "Model 'segment.pt' deployed successfully"
}


# Predict

In [37]:
### 🔹 ทดสอบ Predict Image ###
image_path = "cancer/1.jpg"  # 🔄 เปลี่ยนเป็น path ของรูปที่ต้องการใช้
predict_command = f"predict|{image_path}"
print(f"\n🔍 Sending Predict Command: {predict_command}")
send_command(predict_command)


🔍 Sending Predict Command: predict|cancer/1.jpg
📌 Server Response: {
    "error": "Error processing command: 'str' object has no attribute 'get'"
}


{'error': "Error processing command: 'str' object has no attribute 'get'"}

In [43]:
# ✅ ตั้งค่า Predict Command
predict_command = {
    "command": "predict",
    "image_path": "cancer/1.jpg"  # เปลี่ยนเป็น path ของรูปที่ต้องการใช้
}

print(f"\n🔍 Sending Predict Command: {json.dumps(predict_command, indent=4)}")
response = send_command(predict_command)



🔍 Sending Predict Command: {
    "command": "predict",
    "image_path": "cancer/1.jpg"
}
📌 Server Response: {
    "status": "success",
    "data": {
        "status": "success",
        "message": "Detection completed.",
        "predict_result": "cancer",
        "confidence_scores": {
            "cancer": [
                0.7477406859397888,
                0.5922791957855225,
                0.28380087018013
            ]
        }
    }
}


# Redis Command Via Command Line

In [ ]:
# redis-cli publish train "train|classify"
# redis-cli publish evaluate "evaluate|detect|./runs/detect/train2/weights/best.pt"
# redis-cli publish deploy "deploy|best.pt"
# redis-cli publish predict "predict|cancer/13.jpg"

In [2]:
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda
